# 09 — UMAP colored by degree

Same features as **07**; color = graph **degree**.


### UMAP with degree as color

**Method:** Same feature construction as **`07`** (direction + log radii + log κ). 2D UMAP embedding; scatter color = **graph degree** from `merged`.

**How to read the output:** Localized high-degree color indicates hubs clustering in this feature/UMAP space (or smooth gradients if degree mixes with geometry). Discrete color scales work better if you bin degree in a follow-up. Low degree everywhere may mean the colormap is dominated by outliers—check `vmin`/`vmax` if you extend the notebook.


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import umap

import dmercator3d_io as dm

merged = dm.load_merged_parquet(Path("cache/merged.parquet"))
rng = np.random.default_rng(44)
n = min(8000, len(merged))
idx = rng.choice(len(merged), size=n, replace=False)
sub = merged.iloc[idx].reset_index(drop=True)
U = dm.normalize_direction_nd(sub)
X = np.hstack([U, np.log1p(sub["Inf.Hyp.Rad"]).to_numpy()[:, None], np.log1p(sub["Inf.Kappa"]).to_numpy()[:, None]])
red = umap.UMAP(n_neighbors=30, min_dist=0.1, random_state=44).fit_transform(X)
deg = sub["degree"].to_numpy()
plt.figure(figsize=(6, 5))
sc = plt.scatter(red[:, 0], red[:, 1], s=4, alpha=0.45, c=deg, cmap="magma")
plt.colorbar(sc, label="degree")
plt.title("UMAP colored by degree")
plt.tight_layout()
plt.show()
